<a href="https://colab.research.google.com/github/jrgreen7/SYSC4906/blob/master/W2025/Tutorials/T8/Tutorial-8_Seq2Seq.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 8 - seq2seq - Subtraction

**Semester:** Winter 2026

**Adapted by:** [Kevin Dick](https://kevindick.ai/), [Igor Bogdanov](igorbogdanov@cmail.carleton.ca)

**Adapted from:** [seq2seq Tutorial](https://github.com/lukas/ml-class/blob/master/videos/seq2seq/train.py) originally from this [Keras Blog](https://blog.keras.io/a-ten-minute-introduction-to-sequence-to-sequence-learning-in-keras.html).

---

## PART II: seq2seq LSTM Model: Subtraction

The canonical example of a  sequence-to-sequence (`seq2seq`) learning task is **language translation**. A sequence representing a sentence in one language is encoded into a latent space (an embedded representation) and then decoded into another language.

**Neither fixed input/output length:** The input of characters of variable length from a given alphabet needs to somehow be converted into an out variable in length and possibly from an altogether different alphabet.

`seq2seq` models generally require **massive amounts** of data to learn their task effectively and this tutorial focuses on a unique example that allows the generation of large amounts of data:

### Method: 

We will generate 50 thousands of **string**-representation of math questions (e.g., `"39+3"`) and their target **string**-representation answers (e.g., `"42"`). These will be vectorized and used to train an LSTM model that will learn the "translation" task of converting a query string from "question-language" into a target string in "answer-language"!


This second implementation demonstrates **subtraction** and permits **variable length** inputs.

### Key differences from Part I:
 
1. **Operation**: Subtraction instead of addition
2. **Character set**: Includes the minus sign "-" in addition to digits, plus, and space
3. **Digits**: Allows up to 5 digits per number (increased from 3)
4. **No reversal**: Input sequences are not reversed in this implementation

This implementation shows how the same seq2seq architecture can be adapted to learn different mathematical operations with minimal changes to the model structure.



## Encoding/Decoding Utility

Utility Class for encoding-decoding characters ('0123456789+ ') into one-hot matrices:

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────

from keras.models import Sequential                      # Linear stack of layers (our model type)
from keras.layers import LSTM, TimeDistributed, RepeatVector, Dense  # All layer types used in the seq2seq model
import numpy as np                                       # Array math — used everywhere


# ── Character Lookup Table ────────────────────────────────────────────────────
# Identical to Part 1(a) except this instance will be initialized with a
# larger alphabet: '0123456789+- ' (13 chars) instead of '0123456789+ ' (12 chars)
# to accommodate negative answers from subtraction (e.g., '3-9 = -6').

class CharacterTable(object):
    """Given a set of characters:
    + Encode them to a one hot integer representation
    + Decode the one hot integer representation to their character output
    + Decode a vector of probabilities to their character output
    """

    def __init__(self, chars):
        """Initialize character table.
        # Arguments
            chars: Characters that can appear in the input.
        """
        # Sort and deduplicate so the mapping is deterministic across runs
        self.chars = sorted(set(chars))

        # char → index:  e.g., {'-': 0, '0': 1, ..., '+': 11, ' ': 12}
        # Note: '-' now appears as a legitimate character (negative sign in answers)
        self.char_indices = dict((c, i) for i, c in enumerate(self.chars))

        # index → char:  the reverse lookup
        self.indices_char = dict((i, c) for i, c in enumerate(self.chars))

    def encode(self, C, num_rows):
        """One hot encode given string C.
        # Arguments
            num_rows: Number of rows in the returned one hot encoding. This is
                used to keep the # of rows for each data the same.
        """
        # Start with an all-zero matrix of shape (num_rows, vocab_size)
        # Each row will hold the one-hot vector for one character
        x = np.zeros((num_rows, len(self.chars)))

        # For each character in the string, flip its column to 1
        # e.g., '-' → row i becomes [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
        for i, c in enumerate(C):
            x[i, self.char_indices[c]] = 1

        # Rows beyond len(C) stay all-zero (implicit padding)
        return x

    def decode(self, x, calc_argmax=True):
        if calc_argmax:
            # Each row is either a one-hot vector or a softmax probability
            # distribution — argmax picks the index of the highest value,
            # i.e., the most likely character at each timestep
            x = x.argmax(axis=-1)

        # Map each index back to its character and join into a plain string
        # e.g., [0, 6, 12, 12] → '-6  '
        return "".join(self.indices_char[x] for x in x)


## Defining Dataset Parameters

In [ ]:
# ── Part 1(a) Constants (carried over, unchanged) ─────────────────────────────

TRAINING_SIZE = 50000   # Total number of addition examples (same as Part 1(a))
DIGITS        = 3       # Max digits per number for the ADDITION model (kept for reference)

# A trick from the original Google seq2seq paper.
# Reversing the input brings the first characters of the input closer in the sequence
# to the first characters of the output, which makes the gradient signal stronger during backpropagation.
REVERSE = True

# Maximum length of input is 'int + int' (e.g., '345+678'). Maximum length of int is DIGITS.
MAXLEN = DIGITS + 1 + DIGITS   # = 7, inherited from Part 1(a) — NOT used by the subtraction model


# ── Part 1(b) Parameters (subtraction-specific) ───────────────────────────────

__digits        = 5                          # Max digits per number — increased from 3 to 5.
                                             # Allows numbers up to 99999, making the task harder.
__training_size = 50000                      # Same dataset size as Part 1(a).
                                             # Double underscores avoid collision with Part 1(a)'s globals.

maxlen = __digits + 1 + __digits             # = 11 (e.g., '99999-99999')
                                             # Longer than Part 1(a)'s MAXLEN=7 due to 5-digit numbers.

# All the numbers, plus sign, MINUS sign, and space for padding.
# Key difference from Part 1(a): '-' is added to the alphabet.
# It serves double duty — as the subtraction operator in questions (e.g., '42-7')
# AND as a negative sign in answers (e.g., '3-9' → '-6').
chars  = "0123456789+- "                     # 13 characters vs Part 1(a)'s 12

# Sort and deduplicate so the mapping is deterministic across runs
ctable = CharacterTable(chars)

# Empty containers — filled by the generation loop below
questions = []
expected  = []
seen      = set()   # Tracks (a, b) pairs already generated to avoid duplicates

print(f"Model parameters initialized:")
print(f"- Training size: {__training_size} examples")
print(f"- Maximum digits per number: {__digits}")
print(f"- Input reversal: {REVERSE}")                  # Still True — same reversal trick applies
print(f"- Maximum input length: {maxlen} characters")  # 11, not 7
print(f"- Character set: '{chars}'")                   # 13 chars, not 12
print("Ready to generate training data...")

## Dataset Generation: Subtraction Problems

In [ ]:
# Keep generating examples until we hit our target dataset size (50,000)
print("Generating data...")
while len(questions) < __training_size:

    # Lambda that builds one random integer:
    #   - picks a random number of digits: between 1 and __digits (e.g., 1 to 5)
    #   - for each digit slot, samples a random character from '0123456789'
    #   - joins them into a string, then casts to int (removes any leading zeros)
    #   Example outputs: 7, 42, 819, 54321
    f = lambda: int(
        "".join(
            np.random.choice(list("0123456789"))
            for i in range(np.random.randint(1, __digits + 1))
        )
    )

    # Generate two independent random numbers
    a, b = f(), f()

    # Deduplicate: since 3-9 and 9-3 are treated as the same problem pair, sort before hashing.
    # If we've already seen this (a, b) pair in either order, skip it.
    # NOTE: unlike addition, 3-9 ≠ 9-3 in value (-6 vs +6), but we still
    # deduplicate on the pair to avoid near-duplicate training examples.
    key = tuple(sorted((a, b)))
    if key in seen:
        continue
    seen.add(key)   # Mark this pair as used

    # Build the raw question string, e.g., '42-7'
    # Key difference from Part 1(a): operator is '-' instead of '+'
    q = "{}-{}".format(a, b)

    # Pad with trailing spaces so every question is exactly maxlen=11 characters.
    # e.g., '42-7' (len=4) → '42-7       ' (len=11)
    query = q + " " * (maxlen - len(q))

    # Compute the correct answer as a string.
    # Key difference from Part 1(a): answers can be NEGATIVE (e.g., 3-9 → '-6')
    # which is why '-' was added to the character alphabet.
    ans = str(a - b)

    # Pad the answer with trailing spaces to always be __digits+1=6 characters.
    # e.g., '-6' → '-6    '
    # Note: one extra character slot needed vs Part 1(a) because of the potential '-' sign
    ans += " " * (__digits + 1 - len(ans))

    # No REVERSE step here — unlike Part 1(a), input is NOT reversed in this implementation
    questions.append(query)    # Store the padded question
    expected.append(ans)       # Store the padded answer


# ── Data Summary ──────────────────────────────────────────────────────────────

print("Total addition questions:", len(questions))

# Print the first 5 examples — shown without padding for readability
print("\nSample subtraction data (first 5 examples):")
print("Question | Expected Answer")
print("-" * 40)
for i in range(5):
    print(f"{questions[i].strip()} | {expected[i].strip()}")

# Compute per-example string lengths (after stripping padding) for diagnostics.
# Useful for sanity-checking that data generation worked as expected.
print("\nSubtraction Data Statistics:")
question_lengths = [len(q.strip()) for q in questions]
answer_lengths   = [len(a.strip()) for a in expected]

# Average lengths tell you the typical difficulty of examples in the dataset
print(f"Average question length: {sum(question_lengths)/len(question_lengths):.2f} characters")
print(f"Average answer length:   {sum(answer_lengths)/len(answer_lengths):.2f} characters")

# Min/max sanity checks — shortest should be '1-1', longest should be '99999-99999'
print(f"Shortest question: {min(question_lengths)} characters")
print(f"Longest question:  {max(question_lengths)} characters")

## Converting QA Dataset to NumPy Array

In [ ]:
print("Vectorization...")

# Allocate the input tensor: one matrix per question, filled with zeros.
# Shape: (50000, 11, 13) → (num_examples, sequence_length, vocab_size)
# dtype=bool saves memory — values are only ever 0 or 1 (one-hot)
# Key difference from Part 1(a): 11 timesteps instead of 7 (5-digit numbers),
# and 13 features instead of 12 (extra '-' character in the alphabet)
x = np.zeros((len(questions), maxlen, len(chars)), dtype=bool)

# Allocate the output tensor: one matrix per answer.
# Shape: (50000, 6, 13) → (num_examples, max_answer_length, vocab_size)
# Key difference from Part 1(a): 6 instead of 4 — answers can be up to 6 characters
# long because 5-digit subtraction can produce a 6-character string e.g., '-99998'
y = np.zeros((len(questions), __digits + 1, len(chars)), dtype=bool)

# Fill x: encode each padded question string into its one-hot matrix.
# ctable.encode() returns a (maxlen=11, 13) matrix; we slot it into row i of x.
for i, sentence in enumerate(questions):
    x[i] = ctable.encode(sentence, maxlen)

# Fill y: same process for the answers, but rows are only 6 characters long.
for i, sentence in enumerate(expected):
    y[i] = ctable.encode(sentence, __digits + 1)

## Preparing the Dataset for Training the Model

In [ ]:
# Shuffle x and y TOGETHER using a shared index array.
# This is the standard safe way to shuffle paired arrays in NumPy —
# if you shuffled x and y separately, each answer would be matched
# to the wrong question, silently corrupting the entire dataset.
#
# Why shuffle at all? Data was generated sequentially, so the end of
# the array is biased toward larger numbers (e.g., 5-digit problems).
# Without shuffling, the validation split (last 10%) would be almost
# entirely large-digit examples — an unrepresentative validation set.
indices = np.arange(len(y))    # [0, 1, 2, ..., 49999]
np.random.shuffle(indices)     # e.g., [8312, 441, 27003, ...]
x = x[indices]                 # Reorder x rows using the shuffled index
y = y[indices]                 # Reorder y rows using the SAME shuffled index
                               # → x[i] and y[i] are still a matched pair


# ── Train / Validation Split ──────────────────────────────────────────────────

# Hold out the last 10% of examples strictly for validation.
# The model NEVER trains on these — they exist only to measure
# how well the model generalises to questions it hasn't seen.
#
# e.g., len(x)=50000 → split_at=45000
#   x_train = x[0:45000]   (90%)
#   x_val   = x[45000:]    (10%)
split_at = len(x) - len(x) // 10
(x_train, x_val) = x[:split_at], x[split_at:]
(y_train, y_val) = y[:split_at], y[split_at:]

## Assembling the Model

In [ ]:
# ── Model Architecture ────────────────────────────────────────────────────────
# Identical encoder-decoder structure to Part I.
# Only the input/output dimensions differ due to the larger alphabet and
# longer sequences. Double-underscore names avoid collision with Part 1(a)'s globals.

__hidden_size = 128    # Dimensionality of the RNN's internal state vector — unchanged
__batch_size  = 128    # Number of examples processed per gradient update — unchanged

model = Sequential()

# ENCODER: reads the full 11-character question, collapses it into a single 128-dim vector.
# input_shape=(11, 13): 11 timesteps × 13 chars (vs Part 1(a)'s (7, 12))
model.add(LSTM(__hidden_size, input_shape=(maxlen, len(chars))))
# Output shape after this layer: (batch_size, 128) — the time axis is GONE.

# BRIDGE: copies the single context vector __digits+1=6 times (vs Part 1(a)'s 4)
# to give the decoder one copy per output character position.
model.add(RepeatVector(__digits + 1))
# Output shape after this layer: (batch_size, 6, 128)

# DECODER: unrolls the repeated context into an output sequence.
# return_sequences=True returns ALL 6 hidden states, not just the last one.
model.add(LSTM(__hidden_size, return_sequences=True))
# Output shape after this layer: (batch_size, 6, 128)

# OUTPUT LAYER: applies the same Dense(13) classifier independently to each of
# the 6 timesteps. softmax converts raw scores into a probability distribution
# over the 13-character vocabulary (vs Part 1(a)'s 12).
model.add(TimeDistributed(Dense(len(chars), activation="softmax")))
# Output shape after this layer: (batch_size, 6, 13)

# categorical_crossentropy: standard loss for multi-class classification.
# adam: adaptive learning rate optimiser — robust and fast without manual tuning.
# accuracy: character-level accuracy across all 6 output positions.
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

model.summary()

# Explicit diff summary vs Part 1(a) — useful when comparing training curves later
print("\nComparison with Addition Model:")
print("1. Operation: Subtraction instead of addition")
print(f"2. Character set: '{chars}' (includes minus sign)")   # 13 chars vs 12
print(f"3. Maximum digits: {__digits} (increased from {DIGITS})")  # 5 vs 3
print("4. No input reversal in this implementation")           # REVERSE not applied

## Training the Model 

In [ ]:
print("\nTraining Parameters:")
print(f"- Batch size: {__batch_size} examples")
# Note: no early stopping in this version — runs for a fixed 50 iterations.
# No patience/min_delta logic like Part 1(a) — simpler training loop.

# One iteration = one full pass through all 45,000 training examples.
# We manually loop (instead of epochs=50) so we can display live predictions
# after every single epoch.
for iteration in range(1, 50):
    print()
    print("-" * 50)
    print("Iteration", iteration)

    # Train for exactly one epoch — epochs=1 is intentional: we need control
    # back after each pass to print predictions manually.
    # validation_data is passed so Keras computes val_loss/val_accuracy
    # at the end of the epoch without us having to call model.evaluate() separately.
    model.fit(
        x_train,
        y_train,
        batch_size=__batch_size,    # Process 128 examples per gradient update
        epochs=1,
        validation_data=(x_val, y_val),
    )

    # ── Live Prediction Display ───────────────────────────────────────────────
    # Pick 3 random validation examples and show what the model currently thinks.
    # Fewer examples than Part 1(a)'s 10 — just a quick directional sanity check.
    print("\nExample predictions:")
    correct_count = 0
    print("Question | Expected | Prediction | Result")
    print("-" * 50)

    for i in range(10):
        # Sample one random validation example by index
        ind  = np.random.randint(0, len(x_val))
        rowx = x_val[np.array([ind])]    # Shape (1, 11, 13) — model needs batch dim
        rowy = y_val[np.array([ind])]    # Shape (1,  6, 13)

        # Run a forward pass — no gradient computation, just inference
        preds = model.predict(rowx, verbose=0)    # Shape (1, 6, 13) — softmax probs

        # Decode all three from one-hot / probability matrices back to strings
        q       = ctable.decode(rowx[0])                      # e.g., '42-7       ' (stored form)
        correct = ctable.decode(rowy[0])                      # e.g., '35    '      (true answer)
        guess   = ctable.decode(preds[0], calc_argmax=True)   # e.g., '35    '      (model's answer)

        # Remove padding spaces for clean display — no reversal needed here
        # since REVERSE=False in this implementation
        q_clean       = q.strip()
        correct_clean = correct.strip()
        guess_clean   = guess.strip()

        # Compare stripped strings — padding must not affect the correctness check
        result = "✓" if correct_clean == guess_clean else "✗"
        if correct_clean == guess_clean:
            correct_count += 1

        print(f"{q_clean:12} | {correct_clean:8} | {guess_clean:10} | {result}")

    # Accuracy over these 10 random samples — noisier than Part 1(a)'s 10-sample check
    # but still directionally useful for tracking learning progress
    print(f"\nAccuracy on sample: {correct_count/10:.0%}")

# Takeaway Messages
* The cannonical example of a `seq2seq` learning task is **language translation**: a seqence represening a sentence in one language is encoded into a latent space (an embedded representation) and then decoded into another language.
* In translation, the **input of characters of variable length** and from a **given alphabet** needs to be converted into an **output also variable in length** and possibly from an altogether **different alphabet**.
* `seq2seq` models generally require **massive amounts** of data to effectively learn their task.